# Extracción de keypoints faciales con MediaPipe

Importación de paquetes

In [1]:
import os

import cv2
import mediapipe as mp
import numpy as np


Variables del dataset

In [2]:
DATASET_DIR = "dataset_processed"

CLASSES = sorted(os.listdir(os.path.join(DATASET_DIR, "train")))
print("Clases:", CLASSES)


Clases: ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']


Índices de landmarks y features geométricas

In [ ]:
GEOMETRIC_FEATURE_NAMES = [
    "ear_a",
    "ear_b",
    "mar",
    "mouth_width_norm",
    "eyebrow_raise_a",
    "eyebrow_raise_b",
    "inter_eyebrow",
    "left_corner_lift",
    "right_corner_lift",
]


def _dist(p1, p2):
    return float(np.linalg.norm(np.asarray(p1) - np.asarray(p2)))


def compute_geometric_features(landmarks_px):
    """landmarks_px: array (478, 2) con las coordenadas en píxeles de cada punto."""

    def pt(i):
        return landmarks_px[i]

    iod = _dist(pt(33), pt(263))

    ear_a = (_dist(pt(160), pt(144)) + _dist(pt(158), pt(153))) / (2 * _dist(pt(33), pt(133)))
    ear_b = (_dist(pt(385), pt(380)) + _dist(pt(387), pt(373))) / (2 * _dist(pt(362), pt(263)))

    mar = _dist(pt(13), pt(14)) / _dist(pt(61), pt(291))
    mouth_width_norm = _dist(pt(61), pt(291)) / iod

    eyebrow_raise_a = _dist(pt(105), pt(158)) / iod
    eyebrow_raise_b = _dist(pt(334), pt(387)) / iod
    inter_eyebrow = _dist(pt(55), pt(285)) / iod

    mouth_mid_y = (pt(13)[1] + pt(14)[1]) / 2
    left_corner_lift = (mouth_mid_y - pt(61)[1]) / iod
    right_corner_lift = (mouth_mid_y - pt(291)[1]) / iod

    return np.array(
        [
            ear_a,
            ear_b,
            mar,
            mouth_width_norm,
            eyebrow_raise_a,
            eyebrow_raise_b,
            inter_eyebrow,
            left_corner_lift,
            right_corner_lift,
        ],
        dtype=np.float32,
    )


Extracción por split

In [4]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

landmarker_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
)
landmarker = FaceLandmarker.create_from_options(landmarker_options)


def extract_features_for_split(split_name):
    feature_rows, label_rows, filename_rows = [], [], []
    n_fail = 0

    split_dir = os.path.join(DATASET_DIR, split_name)
    for label in CLASSES:
        class_dir = os.path.join(split_dir, label)
        for fname in sorted(os.listdir(class_dir)):
            image_bgr = cv2.imread(os.path.join(class_dir, fname))
            if image_bgr is None:
                n_fail += 1
                continue

            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            h, w = image_rgb.shape[:2]

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            result = landmarker.detect(mp_image)
            if not result.face_landmarks:
                n_fail += 1
                continue

            landmarks_px = np.array(
                [[lm.x * w, lm.y * h] for lm in result.face_landmarks[0]],
                dtype=np.float32,
            )

            feature_rows.append(compute_geometric_features(landmarks_px))
            label_rows.append(label)
            filename_rows.append(fname)

    print(f"{split_name}: {len(feature_rows)} caras procesadas, {n_fail} fallos de MediaPipe")
    return (
        np.array(feature_rows, dtype=np.float32),
        np.array(label_rows),
        np.array(filename_rows),
    )


Guardar los keypoints en disco

In [5]:
for split_name in ("train", "test"):
    features, labels, filenames = extract_features_for_split(split_name)
    out_path = os.path.join(DATASET_DIR, f"{split_name}_keypoints.npz")
    np.savez(
        out_path,
        features=features,
        labels=labels,
        filenames=filenames,
        feature_names=np.array(GEOMETRIC_FEATURE_NAMES),
    )
    print(f"Guardado en {out_path}")

landmarker.close()


train: 36502 caras procesadas, 2598 fallos de MediaPipe
Guardado en dataset_processed\train_keypoints.npz
test: 9182 caras procesadas, 643 fallos de MediaPipe
Guardado en dataset_processed\test_keypoints.npz
